# Stage 1 -- Data, label structure and split protocols (E0)
Loads DIVE (Kaggle dataset `dive-smart-contracts`, or Zenodo fallback), verifies the
schema, computes the label statistics and co-occurrence matrix used by LGC, builds the
three split protocols (S1 random / S2 compiler-shift / S3 temporal) and publishes them
for all later stages. Outputs: results/, plots/ (PDF+PNG), logs/, tables/, splits.csv.
**Attach:** `scvd-lib`, `dive-smart-contracts` (optional). **Accelerator:** none needed.

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage1_eda')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

In [ ]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

In [ ]:
card = lab[scvd.LABELS].sum(1)
stats = pd.DataFrame({'label': scvd.LABELS, 'short': scvd.LABEL_SHORT,
                      'positive': [int(lab[c].sum()) for c in scvd.LABELS],
                      'rate': [lab[c].mean() for c in scvd.LABELS]})
stats['labelset_size_mean'] = card.mean(); stats['density'] = card.mean() / 8
stats.to_csv(f'{root}/results/stage1_label_stats.csv', index=False)
open(f'{root}/tables/stage1_label_stats.tex', 'w').write(
    stats[['short', 'positive', 'rate']].to_latex(index=False, float_format='%.4f'))
log.info('label cardinality=%.3f density=%.4f', card.mean(), card.mean() / 8)
stats

In [ ]:
scvd.plot_label_prevalence(lab, root, log)
M = scvd.plot_cooccurrence(lab, root, log)
pd.DataFrame(M, index=scvd.LABEL_SHORT, columns=scvd.LABEL_SHORT).to_csv(
    f'{root}/results/stage1_cooccurrence.csv')

In [ ]:
splits = scvd.build_splits(pre, lab, seed=0)
p = scvd.save_splits(root, splits, pre, lab)
sizes = {s: {'train': len(v[0]), 'val': len(v[1]), 'test': len(v[2])} for s, v in splits.items()}
json.dump(sizes, open(f'{root}/results/stage1_split_sizes.json', 'w'), indent=1)
scvd.plot_split_sizes(sizes, root, log)
log.info('split sizes %s', sizes)

In [ ]:
url = scvd.push_dataset(root, 'stage1_eda', 'scvd-stage1-eda', 'SCVD Stage 1: DIVE EDA, label statistics and splits', log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)